# Capítulo 5 · Generación aumentada con recuperación (RAG)

El primer asistente que responde con los documentos de Ciclos Norte: recuperar los fragmentos más parecidos a la pregunta, construir la instrucción y generar la respuesta con citas. Lo probamos con veinte preguntas, quince con respuesta en los documentos y cinco sin ella, y lo mejoramos con ejemplos y con una comprobación previa.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*. El cuaderno carga dos modelos: el de lenguaje (Qwen2.5-1.5B-Instruct con GPU, Qwen3-0.6B sin ella) y el de embeddings del capítulo 4.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

📘 El libro, en Amazon: [tapa blanda](https://www.amazon.es/dp/B0HMXCBCTN) · [Kindle](https://www.amazon.es/dp/B0HMXG3NXM)

## 1. Los modelos y la base de conocimiento

In [ ]:
"""Capítulo 5: generación aumentada con recuperación (RAG). En GPU o CPU."""
import json
import re
import time
import urllib.request
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen2.5-1.5B-Instruct"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-0.6B"             # sin GPU: más pequeño y más rápido
MODELO = None                              # None: elegir según el entorno
MODELO_EMBEDDINGS = "intfloat/multilingual-e5-base"
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/"



def cargar(nombre=None):
    """El modelo de lenguaje y el de embeddings, en la GPU si la hay."""
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    if nombre is None:
        nombre = MODELO_GPU if dispositivo == "cuda" else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(nombre, dtype=torch.float32)
    modelo.to(dispositivo).eval()
    embeddings = SentenceTransformer(MODELO_EMBEDDINGS, device=dispositivo)
    print(f"Modelos {nombre} y {MODELO_EMBEDDINGS} en {dispositivo}")
    return tok, modelo, embeddings


def descargar(archivo):
    """Un archivo de la carpeta datos; en Colab, se descarga del repositorio."""
    ruta = DATOS / archivo
    if not ruta.exists():
        DATOS.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + ruta.as_posix(), ruta)
    return ruta.read_text(encoding="utf-8")


def base_de_conocimiento():
    """Los fragmentos en los que se busca: preguntas frecuentes y secciones de
    las condiciones generales, cada uno con su fuente."""
    faq = json.loads(descargar("preguntas_frecuentes.json"))["preguntas"]
    fragmentos = [{"fuente": f"Preguntas frecuentes, {p['id']}",
                   "texto": p["pregunta"] + " " + p["respuesta"]} for p in faq]
    condiciones = descargar("condiciones_generales.md")
    for seccion in re.split(r"\n(?=## )", condiciones)[1:]:
        titulo, texto = seccion.strip().split("\n", 1)
        fragmentos.append({"fuente": "Condiciones generales, " + titulo[3:],
                           "texto": texto.strip()})
    return fragmentos

In [ ]:
tok, modelo, embeddings = cargar(MODELO)
fragmentos = base_de_conocimiento()
print(len(fragmentos), "fragmentos. Dos de ellos:")
for f in (fragmentos[16], fragmentos[36]):
    print(f"- ({f['fuente']}) {f['texto'][:150]}...")

## 2. Recuperar

In [ ]:
class Buscador:
    """El buscador semántico del capítulo 4."""

    def __init__(self, fragmentos):
        self.fragmentos = fragmentos
        self.matriz = embeddings.encode(
            ["passage: " + f["texto"] for f in fragmentos], normalize_embeddings=True)

    def buscar(self, consulta, k=3):
        vector = embeddings.encode(["query: " + consulta], normalize_embeddings=True)
        puntos = self.matriz @ vector[0]
        return [self.fragmentos[i] for i in np.argsort(-puntos)[:k]]

In [ ]:
buscador = Buscador(fragmentos)
for f in buscador.buscar("¿Puedo pagar en 12 meses una bici de 900 euros?"):
    print(f["fuente"])

## 3. Construir la instrucción y generar

In [ ]:
SISTEMA_SIN_DOCUMENTOS = ("Eres el asistente de atención al cliente de Ciclos "
                          "Norte, una tienda de bicicletas de Oviedo. Responde "
                          "en español, en dos o tres frases.")


NO_LO_SE = "No tengo esa información; paso tu consulta al equipo."


SISTEMA_RAG = (
    "Eres el asistente de atención al cliente de Ciclos Norte, una tienda de "
    "bicicletas de Oviedo. Responde a la pregunta del cliente en español, en dos "
    "o tres frases, usando solo la información de los documentos que se te dan. "
    "Después de cada dato, indica entre corchetes el número del documento del que "
    "sale, por ejemplo [2]. Si los documentos no contienen la respuesta, responde "
    f"exactamente: «{NO_LO_SE}» No inventes nada.")


def instruccion_rag(pregunta, documentos):
    """El mensaje del usuario: los documentos numerados y la pregunta."""
    lista = "\n\n".join(f"[{i}] ({d['fuente']}) {d['texto']}"
                        for i, d in enumerate(documentos, 1))
    return f"Documentos:\n\n{lista}\n\nPregunta del cliente: {pregunta}"

In [ ]:
pregunta = "¿Puedo pagar en 12 meses una bici de 900 euros?"
print(instruccion_rag(pregunta, buscador.buscar(pregunta)))

In [ ]:
def preguntar(sistema, usuario, ejemplos=(), max_tokens=200):
    """Envía una conversación al modelo y devuelve su respuesta como texto."""
    mensajes = [{"role": "system", "content": sistema}]
    for pregunta, respuesta in ejemplos:
        mensajes += [{"role": "user", "content": pregunta},
                     {"role": "assistant", "content": respuesta}]
    mensajes.append({"role": "user", "content": usuario})
    texto = tok.apply_chat_template(mensajes, tokenize=False,
                                    add_generation_prompt=True,
                                    enable_thinking=False)
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    nuevos = salida[0, entrada["input_ids"].shape[1]:]
    respuesta = tok.decode(nuevos, skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", respuesta, flags=re.S).strip()


def responder(pregunta, buscador, k=3):
    """RAG: recuperar k fragmentos, construir la instrucción y generar."""
    if k == 0:
        return preguntar(SISTEMA_SIN_DOCUMENTOS, pregunta), []
    documentos = buscador.buscar(pregunta, k)
    return preguntar(SISTEMA_RAG, instruccion_rag(pregunta, documentos)), documentos

In [ ]:
for pregunta in ["¿Puedo pagar en 12 meses una bici de 900 euros?",
                 "¿Cuánto pesa la bicicleta Urbana 5?"]:
    print("Pregunta:", pregunta)
    print("  Sin documentos:", responder(pregunta, buscador, k=0)[0])
    print("  Con RAG:", responder(pregunta, buscador)[0], "\n")

## 4. Dos mejoras: ejemplos resueltos y una comprobación previa

In [ ]:
EJEMPLOS_RAG = [
    (instruccion_rag("¿Tenéis cascos para mi hijo de cinco años?", [
        {"fuente": "Catálogo, cascos", "texto": "Cascos infantiles en las tallas "
         "XS (46-50 cm) y S (50-54 cm), recomendados de 3 a 8 años."},
        {"fuente": "Condiciones generales, 3. Precios",
         "texto": "Todos los precios incluyen el IVA."}]),
     "Sí, tenemos cascos infantiles en las tallas XS y S, recomendados de 3 a 8 "
     "años [1]."),
    (instruccion_rag("¿Vendéis patinetes eléctricos?", [
        {"fuente": "Catálogo, bicicletas eléctricas", "texto": "Bicicletas "
         "eléctricas urbanas y de montaña, con autonomía de 60 a 120 km."},
        {"fuente": "Preguntas frecuentes, 99", "texto": "¿Alquiláis bicicletas? "
         "Sí, bicicletas de paseo y eléctricas por días."}]),
     NO_LO_SE),
]


SISTEMA_COMPROBAR = ("Decide si los documentos contienen la información necesaria "
                     "para responder a la pregunta del cliente. Responde solo «sí» "
                     "o «no».")


def responder_con_ejemplos(pregunta, buscador, k=3):
    """RAG con dos ejemplos resueltos: uno que cita y otro que no sabe."""
    documentos = buscador.buscar(pregunta, k)
    instruccion = instruccion_rag(pregunta, documentos)
    return preguntar(SISTEMA_RAG, instruccion, EJEMPLOS_RAG), documentos


def responder_comprobando(pregunta, buscador, k=3):
    """Antes de responder, pregunta al modelo si los documentos bastan."""
    documentos = buscador.buscar(pregunta, k)
    instruccion = instruccion_rag(pregunta, documentos)
    veredicto = preguntar(SISTEMA_COMPROBAR, instruccion, max_tokens=3)
    if veredicto.lower().startswith("no"):
        return NO_LO_SE, documentos
    return preguntar(SISTEMA_RAG, instruccion, EJEMPLOS_RAG), documentos

## 5. Evaluar con veinte preguntas

In [ ]:
# Pregunta y textos que debe contener una respuesta correcta (basta con uno);
# None si la respuesta no está en los documentos.
PREGUNTAS = [
    ("¿Puedo devolver un culote si ya lo he abierto?", ["higiene", "no se admite"]),
    ("¿Cuánto cuesta enviar un pedido de 30 euros a Madrid?", ["4,95"]),
    ("¿Cuántos días tengo para devolver una bici?", ["30 días"]),
    ("Si pago por transferencia, ¿cuándo preparáis el pedido?", ["recib"]),
    ("¿Qué garantía tiene la batería de una bici eléctrica?",
     ["2 años", "dos años"]),
    ("¿Cuánto cuesta una revisión en vuestro taller?", ["35 euros"]),
    ("¿Puedo pagar en 12 meses una bici de 900 euros?", ["sin intereses"]),
    ("¿Qué hago si la caja llega golpeada?", ["albarán"]),
    ("¿Cuánto tardáis en responder a un correo?", ["48 horas"]),
    ("¿Las bicicletas llegan montadas?", ["90"]),
    ("¿Cuánta autonomía tiene una bicicleta eléctrica?",
     ["120 km", "120 kilómetros"]),
    ("¿A qué hora abre la tienda los sábados?", ["10:00", "10 de la mañana"]),
    ("¿De cuánto es la tarjeta regalo más cara?", ["100 euros"]),
    ("Si devuelvo solo una parte del pedido, ¿me devolvéis el envío?",
     ["pedido completo", "todo el pedido", "pedido entero"]),
    ("¿Puedo cancelar un pedido que ya ha salido del almacén?",
     ["no haya salido", "no ha salido", "antes de que salga"]),
    ("¿Cuánto pesa la bicicleta Urbana 5?", None),
    ("¿Tenéis aparcamiento para clientes en la tienda?", None),
    ("¿Organizáis rutas en grupo los domingos?", None),
    ("¿Vendéis piezas de recambio de la marca Shimano?", None),
    ("¿Hacéis descuentos a estudiantes?", None),
]

In [ ]:
def contiene(texto, claves):
    """¿Contiene el texto alguna de las claves?"""
    return any(c.lower() in texto.lower() for c in claves)


def evaluar(buscador, forma):
    """Responde a las 20 preguntas y anota qué sale bien y qué sale mal."""
    filas = []
    for pregunta, claves in PREGUNTAS:
        inicio = time.time()
        respuesta, documentos = forma(pregunta, buscador)
        se_abstiene = NO_LO_SE[:25].lower() in respuesta.lower()
        fila = {"pregunta": pregunta, "respuesta": respuesta,
                "fuentes": [d["fuente"] for d in documentos],
                "segundos": time.time() - inicio, "se_abstiene": se_abstiene}
        if claves is None:                      # no está en los documentos
            fila["correcta"] = se_abstiene
        else:
            fila["correcta"] = contiene(respuesta, claves) and not se_abstiene
            fila["recuperada"] = any(contiene(d["texto"], claves) for d in documentos)
            fila["cita"] = bool(re.search(r"\[\d\]", respuesta))
        filas.append(fila)
    return filas


def resumen(filas):
    """Aciertos con respuesta, abstenciones correctas y fallos de recuperación."""
    con = [f for f in filas if "recuperada" in f]
    sin = [f for f in filas if "recuperada" not in f]
    return {"correctas": sum(f["correcta"] for f in con),
            "recuperadas": sum(f["recuperada"] for f in con),
            "abstenciones_indebidas": sum(f["se_abstiene"] for f in con),
            "con_citas": sum(f["cita"] for f in con),
            "abstenciones_correctas": sum(f["correcta"] for f in sin),
            "segundos": round(sum(f["segundos"] for f in filas) / len(filas), 1)}

In [ ]:
FORMAS = {
    "Sin documentos": lambda p, b: responder(p, b, k=0),
    "RAG, 1 fragmento": lambda p, b: responder(p, b, k=1),
    "RAG, 3 fragmentos": lambda p, b: responder(p, b, k=3),
    "RAG, 5 fragmentos": lambda p, b: responder(p, b, k=5),
    "Con ejemplos": responder_con_ejemplos,
    "Comprobando antes": responder_comprobando,
}

In [ ]:
resultados = {}
for nombre, forma in FORMAS.items():
    resultados[nombre] = evaluar(buscador, forma)
    print(f"{nombre}:", resumen(resultados[nombre]))

Lee las respuestas de una variante, con una marca en las incorrectas:

In [ ]:
variante = "Con ejemplos"
for fila in resultados[variante]:
    marca = "  " if fila["correcta"] else "✗ "
    print(marca + fila["pregunta"])
    print("     ", fila["respuesta"][:200])

## Ejercicios

1. Explica con tus palabras las tres fases de un sistema RAG. ¿Qué parte del sistema es responsable de cada uno de los tres tipos de fallo del capítulo?

2. Para cada una de las quince preguntas con respuesta, mira qué fragmentos se recuperan con k = 3. ¿En cuál falla la recuperación? ¿Por qué crees que el buscador no encuentra el fragmento correcto?

3. Escribe cinco preguntas más cuya respuesta no esté en los documentos y evalúa las variantes con ejemplos y con comprobación. ¿Se mantienen las cifras del capítulo?

4. Cambia la frase fija por otra más natural, como «Lo siento, no lo sé». ¿Cambia cuántas veces se abstiene el modelo?

5. En la variante con comprobación, cambia el mensaje de sistema de la comprobación para que sea menos estricto, por ejemplo: «Responde no solo si ningún documento trata del tema de la pregunta». Comprueba que obtienes un resultado parecido al del capítulo. Prueba después otras dos redacciones. ¿Cuál equilibra mejor inventar y callar?

6. Escribe una función que compruebe las citas: que cada número entre corchetes corresponda a un documento recuperado y que ese documento contenga alguna de las claves de la pregunta. ¿Cuántas citas son correctas?

7. Añade a la base de conocimiento un fragmento falso: «El envío a Canarias es gratuito en todos los pedidos». Pregunta cuánto cuesta enviar a Canarias. ¿Qué responde el asistente? ¿Qué te dice esto sobre la calidad de los documentos?

8. Mide cuántos tokens tiene la instrucción con 1, 3 y 5 fragmentos. ¿Cuánto crece? ¿Qué pasaría con fragmentos de 320 palabras?

9. Cambia el orden de los fragmentos para que el más parecido a la pregunta vaya el último, junto a la pregunta, en lugar del primero. ¿Cambian los aciertos?

10. **Asistente de preguntas.** Escribe una función `asistente(pregunta)` que use la mejor variante para ti y muestre la respuesta seguida de las fuentes citadas. Pruébala con diez preguntas tuyas y escribe un breve informe con lo que funciona y lo que no.

## Referencias

Modelos [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) y [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base). RAG: [Lewis y colaboradores (2020)](https://arxiv.org/abs/2005.11401). Lecciones 11.06 y 05.23 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La tienda Ciclos Norte y sus documentos son ficticios. Código y experimentos propios.